### Human in the Loop

In [21]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_groq import ChatGroq
from langchain.messages import HumanMessage
from langgraph.types import Command
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = ChatGroq(model="llama-3.3-70b-versatile")

def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject {subject}"

In [22]:
agent = create_agent(
    model=model,
    tools = [read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve","edit","reject"]
                },
                "read_email_tool":False,
            }
        )
    ]
)

In [23]:
config = {"configurable": {
    "thread_id": "test-approve"
}}

result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@example.com with a subject 'Hello' and body 'How are you?'")]},
    config=config
)

In [24]:
result

{'messages': [HumanMessage(content="Send email to john@example.com with a subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='12980347-5aea-42cc-b0a0-efb44c3ffc3e'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '5ah3czv0s', 'function': {'arguments': '{"body":"How are you?","recipient":"john@example.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 310, 'total_tokens': 342, 'completion_time': 0.060465506, 'completion_tokens_details': None, 'prompt_time': 0.03094675, 'prompt_tokens_details': None, 'queue_time': 0.164078198, 'total_time': 0.091412256}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_3272ea2d91', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ea82d-faa8-7380-a643-bac2098abe79-0', tool_calls=[{'name': 'send_email_tool', 'arg

In [25]:
if "__interrupt__" in result: 
    print("Paused Approving...")

    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "approve"}
                ]
            }
        ),
        config=config
    )

    print(f"Result: {result['messages'][-1].content}")

Paused Approving...
Result: The email has been sent.


In [ ]:
result

# similarly fr reject and edit

{'messages': [HumanMessage(content="Send email to john@example.com with a subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='12980347-5aea-42cc-b0a0-efb44c3ffc3e'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '5ah3czv0s', 'function': {'arguments': '{"body":"How are you?","recipient":"john@example.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 310, 'total_tokens': 342, 'completion_time': 0.060465506, 'completion_tokens_details': None, 'prompt_time': 0.03094675, 'prompt_tokens_details': None, 'queue_time': 0.164078198, 'total_time': 0.091412256}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_3272ea2d91', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ea82d-faa8-7380-a643-bac2098abe79-0', tool_calls=[{'name': 'send_email_tool', 'arg